# Explain logits and sampling

This notebook builds intuition for how language models turn raw scores (**logits**) into sampled tokens.

We will:

1. Define a tiny vocabulary and a toy set of logits.
2. Convert logits into probabilities with softmax.
3. Explore temperature scaling.
4. Compare greedy decoding, multinomial sampling, top-k sampling, and nucleus/top-p sampling.
5. Run a tiny autoregressive generation loop so the sampling choices are easy to see.


## Setup

The notebook intentionally uses only `numpy` and `matplotlib` so the mechanics are visible and easy to modify.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)


## A toy vocabulary and model output

A language model predicts the next token by producing one raw score per vocabulary item. These raw scores are called **logits**.

Logits are not probabilities:

- They can be negative.
- They do not need to add up to 1.
- Larger logits indicate tokens the model prefers more strongly.


In [ ]:
vocab = np.array(["the", "cat", "sat", "on", "mat", ".", "dog", "ran"])
logits = np.array([3.2, 2.7, 1.1, 0.4, 0.1, -0.2, 1.9, 0.8])

list(zip(vocab, logits))


## Softmax: converting logits to probabilities

The softmax function exponentiates each logit and normalizes the results so they sum to 1.

For numerical stability, we subtract the maximum logit before exponentiating. This does not change the final probabilities.


In [ ]:
def softmax(x):
    shifted = x - np.max(x)
    exp_x = np.exp(shifted)
    return exp_x / exp_x.sum()

probs = softmax(logits)

for token, logit, prob in zip(vocab, logits, probs):
    print(f"{token:>4}  logit={logit:>5.2f}  prob={prob:>6.3f}")

print(f"
Probability sum: {probs.sum():.3f}")


In [ ]:
def plot_distribution(tokens, values, title, ylabel):
    plt.figure(figsize=(8, 3.5))
    bars = plt.bar(tokens, values)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.ylim(0, max(values) * 1.2 if max(values) > 0 else 1)
    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:.2f}",
                 ha="center", va="bottom", fontsize=9)
    plt.show()

plot_distribution(vocab, probs, "Softmax probabilities", "Probability")


## Temperature

Temperature changes how sharp or flat the probability distribution is before sampling.

Given logits `z`, temperature-scaled probabilities are:

```python
softmax(z / temperature)
```

- Low temperature, such as `0.3`, makes the model more confident and repetitive.
- Temperature `1.0` keeps the original softmax distribution.
- High temperature, such as `2.0`, makes lower-probability tokens more likely.


In [ ]:
temperatures = [0.3, 0.7, 1.0, 1.5, 2.0]

temp_probs = {temp: softmax(logits / temp) for temp in temperatures}

fig, axes = plt.subplots(len(temperatures), 1, figsize=(8, 10), sharex=True)
for ax, temp in zip(axes, temperatures):
    ax.bar(vocab, temp_probs[temp])
    ax.set_ylim(0, 1)
    ax.set_ylabel(f"T={temp}")
axes[-1].set_xlabel("Token")
fig.suptitle("Effect of temperature on next-token probabilities", y=0.92)
plt.show()


## Sampling methods

Different decoding strategies turn probabilities into an actual next token.

### Greedy decoding

Always choose the token with the highest probability.

### Multinomial sampling

Sample directly from the full probability distribution.

### Top-k sampling

Keep only the `k` most likely tokens, renormalize, and sample from that smaller set.

### Nucleus / top-p sampling

Keep the smallest set of most likely tokens whose cumulative probability is at least `p`, renormalize, and sample from that set.


In [ ]:
def greedy_decode(tokens, probabilities):
    return tokens[np.argmax(probabilities)]


def multinomial_sample(tokens, probabilities, rng):
    return rng.choice(tokens, p=probabilities)


def top_k_sample(tokens, probabilities, k, rng):
    keep_idx = np.argsort(probabilities)[-k:]
    keep_probs = probabilities[keep_idx]
    keep_probs = keep_probs / keep_probs.sum()
    return rng.choice(tokens[keep_idx], p=keep_probs)


def top_p_sample(tokens, probabilities, p, rng):
    sorted_idx = np.argsort(probabilities)[::-1]
    sorted_probs = probabilities[sorted_idx]
    cumulative = np.cumsum(sorted_probs)
    cutoff = np.searchsorted(cumulative, p) + 1
    keep_idx = sorted_idx[:cutoff]
    keep_probs = probabilities[keep_idx]
    keep_probs = keep_probs / keep_probs.sum()
    return rng.choice(tokens[keep_idx], p=keep_probs)

print("Greedy:", greedy_decode(vocab, probs))
print("Multinomial samples:", [multinomial_sample(vocab, probs, rng) for _ in range(10)])
print("Top-k samples:", [top_k_sample(vocab, probs, k=3, rng=rng) for _ in range(10)])
print("Top-p samples:", [top_p_sample(vocab, probs, p=0.8, rng=rng) for _ in range(10)])


## What top-k and top-p keep

It is useful to inspect which tokens survive the filter before sampling.


In [ ]:
def top_k_table(tokens, probabilities, k):
    keep_idx = np.argsort(probabilities)[-k:][::-1]
    return [(tokens[i], probabilities[i]) for i in keep_idx]


def top_p_table(tokens, probabilities, p):
    sorted_idx = np.argsort(probabilities)[::-1]
    sorted_probs = probabilities[sorted_idx]
    cutoff = np.searchsorted(np.cumsum(sorted_probs), p) + 1
    keep_idx = sorted_idx[:cutoff]
    return [(tokens[i], probabilities[i]) for i in keep_idx]

print("Top-k tokens, k=3")
for token, prob in top_k_table(vocab, probs, k=3):
    print(f"{token:>4}: {prob:.3f}")

print("
Top-p tokens, p=0.8")
for token, prob in top_p_table(vocab, probs, p=0.8):
    print(f"{token:>4}: {prob:.3f}")


## A tiny autoregressive generator

The next cell creates a toy model with a hand-written transition table. It is not a neural network, but it behaves like a language model at decoding time: given the current token, it returns logits for the next token.

This lets us compare decoding strategies in a small, transparent setting.


In [ ]:
transition_logits = {
    "the": np.array([-2.0, 3.1, -0.5, -1.0, -1.0, -2.5, 2.6, -0.8]),
    "cat": np.array([-1.0, -2.0, 3.0, -0.4, -1.5, 0.2, -2.0, 1.5]),
    "dog": np.array([-1.0, -2.0, 0.9, -0.6, -1.5, 0.1, -2.0, 3.0]),
    "sat": np.array([-1.0, -2.0, -2.0, 3.0, -0.5, 0.1, -2.0, -1.0]),
    "ran": np.array([-0.2, -2.0, -1.8, 2.4, -1.0, 0.8, -2.0, -2.0]),
    "on":  np.array([1.8, -1.0, -2.0, -2.0, 2.8, -0.5, -1.0, -2.0]),
    "mat": np.array([-0.4, -2.0, -2.0, -2.0, -2.0, 3.2, -2.0, -2.0]),
    ".":   np.array([3.0, 0.4, -1.2, -1.5, -1.5, -2.0, 0.8, -1.2]),
}


def next_logits(current_token):
    return transition_logits[current_token]


def generate(start="the", steps=12, temperature=1.0, method="multinomial", k=3, p=0.9, seed=0):
    local_rng = np.random.default_rng(seed)
    tokens = [start]
    for _ in range(steps):
        current_logits = next_logits(tokens[-1])
        probabilities = softmax(current_logits / temperature)
        if method == "greedy":
            next_token = greedy_decode(vocab, probabilities)
        elif method == "multinomial":
            next_token = multinomial_sample(vocab, probabilities, local_rng)
        elif method == "top_k":
            next_token = top_k_sample(vocab, probabilities, k=k, rng=local_rng)
        elif method == "top_p":
            next_token = top_p_sample(vocab, probabilities, p=p, rng=local_rng)
        else:
            raise ValueError(f"Unknown method: {method}")
        tokens.append(next_token)
    return " ".join(tokens)

for method in ["greedy", "multinomial", "top_k", "top_p"]:
    print(f"{method:>11}:", generate(method=method, temperature=0.9, seed=4))


## Experiment: temperature and randomness

Try changing `temperature`, `k`, and `p` below.

A good pattern to look for:

- Lower temperature makes output more deterministic.
- Higher temperature makes output more diverse but also less stable.
- Top-k can remove strange low-probability tokens.
- Top-p adapts the candidate set size based on how concentrated the probabilities are.


In [ ]:
settings = [
    {"method": "multinomial", "temperature": 0.4},
    {"method": "multinomial", "temperature": 1.0},
    {"method": "multinomial", "temperature": 1.8},
    {"method": "top_k", "temperature": 1.0, "k": 2},
    {"method": "top_p", "temperature": 1.0, "p": 0.75},
]

for setting in settings:
    text = generate(seed=12, **setting)
    print(f"{setting}:
  {text}
")


## Takeaways

- **Logits** are raw model scores, not probabilities.
- **Softmax** converts logits into a probability distribution.
- **Temperature** controls how sharp or flat the distribution is.
- **Greedy decoding** is deterministic but can be repetitive.
- **Multinomial sampling** uses the full distribution and can be creative or noisy.
- **Top-k sampling** limits sampling to the `k` strongest candidates.
- **Top-p sampling** keeps a variable-size candidate set covering a target amount of probability mass.

These same ideas scale from this toy example to large language models.
